## Fram Strait 2025

This handler converts the FS2025 I-129 seawater CSV record from Zenodo into MARIS-standard NetCDF4. The source table contains station, position, collection date, hydrographic metadata, a provider sample ID, and I-129 values with absolute uncertainties.

The handler is structured to accept further compatible Fram Strait CSV records through `RECORDS`; each input is read and combined into the `SEAWATER` group.

In [ ]:
#| default_exp handlers.fram_strait2025

In [ ]:
#| export
from fastcore.all import *
import pandas as pd
import numpy as np
import requests
import io

from marisco.callbacks import (
    PerGroupCB, Transformer, EncodeTimeCB,
    SanitizeLonLatCB, RemapCB, AddSampleIDCB
)
from marisco.metadata import (
    GlobAttrsFeeder, BboxCB, DepthRangeCB,
    TimeRangeCB, KeyValuePairCB
)
from marisco.encoders import NetCDFEncoder
from marisco.nc2csv import to_csv

In [ ]:
#| exports
RECORDS = {
    "FS2025_i129": {
        "url": "https://zenodo.org/records/20761832/files/FS2025_i129.csv?download=1",
    },
}

fname_out = "Fram_Strait_2025.nc"
src_dir = None

In [ ]:
#| export
def load_data(
    recs=None,  # Optional record mapping; defaults to all RECORDS
) -> dict:
    "Fetch Fram Strait CSV records and return one combined SEAWATER DataFrame."
    recs = recs or RECORDS
    parts = []

    for record in recs.values():
        resp = requests.get(record["url"], timeout=60)
        resp.raise_for_status()

        df = pd.read_csv(
            io.BytesIO(resp.content),
            encoding="utf-8-sig",
        )
        parts.append(df)

    return {"SEAWATER": pd.concat(parts, ignore_index=True)}

In [ ]:
#| eval: false
dfs = load_data()

In [ ]:
#|eval: false
print(dfs['SEAWATER'].describe(include='number').T[['count', 'mean', 'min', 'max']])

                   count          mean           min           max
Station            177.0  3.723107e+02  3.410000e+02  4.150000e+02
Latitude_degN      177.0  7.895424e+01  7.883217e+01  8.040850e+01
Longitude_degE     177.0 -4.121274e+00 -1.340367e+01  8.000000e+00
Niskin             177.0  1.021469e+01  1.000000e+00  2.300000e+01
Pressure_dbar      177.0  2.675278e+02  4.524000e+00  2.621694e+03
PracticalSalinity  177.0  3.379420e+01  2.898850e+01  3.510550e+01
Temperature_degC   177.0  8.848593e-01 -1.815000e+00  9.014500e+00
Sample_ID          177.0  9.009040e+01  1.000000e+00  1.800000e+02
I129_at_l          177.0  3.038391e+09  1.502199e+08  7.150855e+09
unc_I129_at_l      177.0  7.779234e+07  4.745141e+06  1.819380e+08


## Rename and standardise columns

FS2025 is seawater-only. `RenameColsCB` maps provider metadata columns to MARIS working names, `ParseDateTimeCB` converts the collection date to UTC `TIME`, and `AddUnknownDepthCB` records unavailable sample depth as the MARIS sentinel value `-1`.

### I-129 column convention

The provider uses `I129_at_l` for the value in atoms per litre and `unc_I129_at_l` for its absolute uncertainty. This `{nuclide}_at_{unit}` naming pattern is read directly by `ParseNuclideUnitCB` after `MeltCB` reshapes the data to long format; no special column-name normalisation is required.

In [ ]:
#| export
class RenameColsCB(PerGroupCB):
    "Map FS2025 provider columns to MARIS standard names."
    grps = ["SEAWATER"]

    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.rename(columns={
            "Station": "STATION",
            "Latitude_degN": "LAT",
            "Longitude_degE": "LON",
            "PracticalSalinity": "SAL",
            "Temperature_degC": "TEMP",
            "Sample_ID": "SMP_ID_PROVIDER",
        })

In [ ]:
# Verify RenameColsCB maps provider columns to MARIS names
dfs_mock = {
    "SEAWATER": pd.DataFrame({
        "Station": [341],
        "Latitude_degN": [78.832167],
        "Longitude_degE": [-2.004667],
        "PracticalSalinity": [34.9],
        "Temperature_degC": [0.0538],
        "Sample_ID": [1],
    })
}

tfm = Transformer(dfs_mock, cbs=[RenameColsCB()])
tfm()

for col in ["STATION", "LAT", "LON", "SAL", "TEMP", "SMP_ID_PROVIDER"]:
    test_eq(col in tfm.dfs["SEAWATER"].columns, True)

print("RenameColsCB: FS2025 columns mapped correctly. ✓")

RenameColsCB: FS2025 columns mapped correctly. ✓


In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[RenameColsCB()])
tfm()
print(
    tfm.dfs["SEAWATER"][
        ["LAT", "LON", "STATION", "SAL", "TEMP", "SMP_ID_PROVIDER"]
    ].head(2).to_string()
)

         LAT       LON  STATION     SAL    TEMP  SMP_ID_PROVIDER
0  78.832167 -2.004667      341  34.900  0.0538                1
1  78.832167 -2.004667      341  34.917  0.8933                2


In [ ]:
#| export
class ParseDateTimeCB(PerGroupCB):
    "Parse FS2025 collection date into a UTC TIME value."
    grps = ["SEAWATER"]

    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.assign(
            TIME=pd.to_datetime(df["Date"], format="%Y-%m-%d", utc=True)
        ).drop(columns="Date")

In [ ]:
#| export
class AddUnknownDepthCB(PerGroupCB):
    "Set sampling depth to -1 because FS2025 provides pressure, not depth in metres."
    grps = ["SEAWATER"]

    def each_grp(self, grp, df, tfm):
        df["SMP_DEPTH"] = -1.0

In [ ]:
# Verify ParseDateTimeCB parses Date into UTC TIME
dfs_mock = {"SEAWATER": pd.DataFrame({"Date": ["2025-07-30"]})}

tfm = Transformer(dfs_mock, cbs=[ParseDateTimeCB()])
tfm()

test_eq("TIME" in tfm.dfs["SEAWATER"].columns, True)
test_eq("Date" not in tfm.dfs["SEAWATER"].columns, True)
print(f"ParseDateTimeCB: TIME = {tfm.dfs['SEAWATER']['TIME'].iloc[0]}. ✓")

ParseDateTimeCB: TIME = 2025-07-30 00:00:00+00:00. ✓


In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB(),
    AddUnknownDepthCB(),
])
tfm()

print(tfm.dfs["SEAWATER"][["TIME", "SMP_DEPTH"]].head(3).to_string())

                       TIME  SMP_DEPTH
0 2025-07-30 00:00:00+00:00       -1.0
1 2025-07-30 00:00:00+00:00       -1.0
2 2025-07-30 00:00:00+00:00       -1.0


::: {.callout-note}

#### FS2025 pressure and bottle metadata

FS2025 provides `Pressure_dbar`, not sampling depth in metres. The handler therefore sets `SMP_DEPTH` to `-1`, the MARIS sentinel for unavailable depth, rather than treating pressure as depth. `Niskin` identifies the bottle position and is not used as the provider sample ID; `Sample_ID` is preserved as `SMP_ID_PROVIDER`. `Pressure_dbar` and `Niskin` are not included in the NetCDF output.

:::

## Reshape wide to long

FS2025 has one I-129 measurement and one matching uncertainty column per sample. MARIS requires long format: one row per measurement with `NUCLIDE`, `UNIT`, `VALUE`, and `UNC`.

`MeltCB` performs a pure wide-to-long reshape (guarding against a missing `unc_*` column and against row-duplication via a `_SOURCE_ROW` merge key), `ParseNuclideUnitCB` derives `NUCLIDE` and `UNIT` from the melted column name, and `AuditDropNullCB(["VALUE"])` logs and explicitly drops rows without a value.

In [ ]:
#| exports
META_COLS = ["Cruise", "STATION", "SMP_ID_PROVIDER", "LAT", "LON", "TIME", "SMP_DEPTH", "TEMP", "SAL"]
VAL_COLS = ["I129_at_l"]
NUCLIDE_LUT = {"I129": 28}
UNIT_LUT = {"at_l": 12}
STATIC_METADATA = {"LAB": 0}  # 分析機関が判明したら、そのMARIS LAB IDへ変更

In [ ]:
#| export
class MeltCB(PerGroupCB):
    "Reshape wide value/uncertainty columns into long format (Cartesian-product safe)."
    def __init__(self, meta_cols, val_cols, val_name='VALUE', unc_name='UNC', grps=None):
        store_attr(); self.grps = self.grps or ['SEAWATER']

    def each_grp(self, grp, df, tfm):
        df = df.assign(_SOURCE_ROW=df.index)
        vals = df.melt(id_vars=[*self.meta_cols, '_SOURCE_ROW'], value_vars=self.val_cols, var_name='nuclide_raw', value_name=self.val_name)
        if unc_cols := [f'unc_{c}' for c in self.val_cols if f'unc_{c}' in df.columns]:
            uncs = df.melt(id_vars=['_SOURCE_ROW'], value_vars=unc_cols, value_name=self.unc_name)
            uncs['nuclide_raw'] = uncs['variable'].str.removeprefix('unc_')
            vals = vals.merge(uncs[['_SOURCE_ROW', 'nuclide_raw', self.unc_name]], on=['_SOURCE_ROW', 'nuclide_raw'], how='left')
        else:
            vals[self.unc_name] = pd.NA
        tfm.dfs[grp] = vals.drop(columns=['_SOURCE_ROW']).reset_index(drop=True)


class ParseNuclideUnitCB(PerGroupCB):
    "Parse 'nuclide_raw' into NUCLIDE and UNIT columns using rsplit."
    def __init__(self, grps=None): self.grps = grps or ['SEAWATER']

    def each_grp(self, grp, df, tfm):
        parts = df['nuclide_raw'].str.rsplit('_at_', n=1, expand=True)
        tfm.dfs[grp] = df.assign(NUCLIDE=parts[0], UNIT='at_' + parts[1]).drop(columns=['nuclide_raw'])


class AuditDropNullCB(PerGroupCB):
    "Audit and drop rows with missing required values explicitly."
    def __init__(self, cols: list, grps=None):
        store_attr(); self.grps = self.grps or ['SEAWATER']

    def each_grp(self, grp, df, tfm):
        clean_df = df.dropna(subset=self.cols)
        if (n := len(df) - len(clean_df)) > 0:
            msg = f"[Audit] {grp}: Dropping {n} row(s) missing {self.cols}"
            print(msg); tfm.logs.append(msg)
        tfm.dfs[grp] = clean_df.reset_index(drop=True)


class SetValueCB(PerGroupCB):
    "Assign constant metadata dictionary."
    def __init__(self, constants: dict, grps=None):
        store_attr(); self.grps = self.grps or ['SEAWATER']

    def each_grp(self, grp, df, tfm):
        for k, v in self.constants.items(): df[k] = v
        tfm.dfs[grp] = df

In [ ]:
# Verify MeltCB (1:1 merge, no Cartesian explosion) + ParseNuclideUnitCB + AuditDropNullCB
dfs_mock = {
    "SEAWATER": pd.DataFrame({
        "Cruise": ["FS2025", "FS2025"],
        "STATION": [341, 341],
        "SMP_ID_PROVIDER": [1, 2],
        "I129_at_l": [2.264769e9, np.nan],
        "unc_I129_at_l": [5.886864e7, 1.0e6],
    })
}
mock_meta = ["Cruise", "STATION", "SMP_ID_PROVIDER"]
mock_vals = ["I129_at_l"]

tfm = Transformer(dfs_mock, cbs=[MeltCB(mock_meta, mock_vals), ParseNuclideUnitCB(), AuditDropNullCB(["VALUE"])])
tfm()
out = tfm.dfs["SEAWATER"]
test_eq(len(out), 1)  # row with NaN VALUE dropped
test_eq(out["NUCLIDE"].tolist(), ["I129"])
test_eq(out["UNIT"].tolist(), ["at_l"])
print("MeltCB + ParseNuclideUnitCB + AuditDropNullCB: value/uncertainty melted, split, null dropped correctly. ✓")

# Verify MeltCB guard clause: no KeyError when unc_* column is absent
dfs_mock_no_unc = {"SEAWATER": pd.DataFrame({"Cruise": ["FS2025"], "STATION": [341], "SMP_ID_PROVIDER": [1], "I129_at_l": [2.264769e9]})}
tfm = Transformer(dfs_mock_no_unc, cbs=[MeltCB(mock_meta, mock_vals)])
tfm()
out = tfm.dfs["SEAWATER"]
test_eq(out["UNC"].isna().all(), True)
print("MeltCB: missing unc_* column handled without KeyError. ✓")

[Audit] SEAWATER: Dropping 1 row(s) missing ['VALUE']
MeltCB + ParseNuclideUnitCB + AuditDropNullCB: value/uncertainty melted, split, null dropped correctly. ✓
MeltCB: missing unc_* column handled without KeyError. ✓


In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB(),
    AddUnknownDepthCB(),
    MeltCB(META_COLS, VAL_COLS),
    ParseNuclideUnitCB(),
    AuditDropNullCB(["VALUE"]),
])
tfm()

out = tfm.dfs["SEAWATER"]
print(out[["NUCLIDE", "UNIT", "VALUE", "UNC"]].head(6).to_string())

  NUCLIDE  UNIT         VALUE           UNC
0    I129  at_l  2.264769e+09  5.886864e+07
1    I129  at_l  1.951592e+09  5.074921e+07
2    I129  at_l  2.081041e+09  5.406397e+07
3    I129  at_l  2.238436e+09  5.812364e+07
4    I129  at_l  2.355675e+09  6.119206e+07
5    I129  at_l  2.470268e+09  6.420043e+07

## Remap nomenclatures to MARIS identifiers

`ParseNuclideUnitCB` produces the provider strings `I129` and `at_l`. `RemapCB` maps these to the MARIS nuclide and unit identifiers, but on an unrecognised (non-null) value it silently falls back to `default_val` (`0`) rather than raising — a typo or an unannounced new nuclide/unit code would be written to the NetCDF file as `TBD` with no warning. `MultiGateCB` runs immediately before both `RemapCB` calls and validates every non-null `NUCLIDE`/`UNIT` value against `NUCLIDE_LUT`/`UNIT_LUT`, raising `ValueError` naming the offending value(s) and the known keys on any unmapped token. `SetValueCB` assigns the currently available laboratory identifier as static metadata; the handler does not assign an AREA value, because no area field is required for this dataset.

`MultiGateCB` is defined locally in this notebook only; it does not touch `marisco/callbacks.py` (locked pending sign-off — see [`nbs/reference/remapcb-failfast-refactoring.md`](../reference/remapcb-failfast-refactoring.md) for the separate, pending-approval proposal to harden `RemapCB` itself).

In [ ]:
#| export
class MultiGateCB(PerGroupCB):
    "Fail-fast pre-RemapCB gate: every non-null value in each `col_src` must be a known key in its paired LUT."
    def __init__(self,
                 lut_pairs: dict,  # {col_src: lut} pairs to validate before RemapCB resolves them
                 grps: list=None,  # Groups to process (None = all)
                 ):
        store_attr()

    def each_grp(self, grp, df, tfm):
        for col_src, lut in self.lut_pairs.items():
            if unknown := set(df[col_src].dropna()) - set(lut):
                raise ValueError(
                    f"MultiGateCB: {grp}.{col_src} has unmapped value(s) {sorted(map(str, unknown))} "
                    f"not present in the lookup table. Known keys: {sorted(map(str, lut))}"
                )

In [ ]:
# Verify MultiGateCB passes known values through untouched, then RemapCB + SetValueCB run as before
dfs_mock = {
    "SEAWATER": pd.DataFrame({
        "NUCLIDE": ["I129"],
        "UNIT": ["at_l"],
        "VALUE": [2.264769e9],
        "UNC": [5.886864e7],
    })
}

tfm = Transformer(dfs_mock, cbs=[
    MultiGateCB(lut_pairs={"NUCLIDE": NUCLIDE_LUT, "UNIT": UNIT_LUT}),
    RemapCB(lut=NUCLIDE_LUT, col_remap="NUCLIDE", col_src="NUCLIDE"),
    RemapCB(lut=UNIT_LUT, col_remap="UNIT", col_src="UNIT"),
    SetValueCB(constants=STATIC_METADATA),
])
tfm()

out = tfm.dfs["SEAWATER"]
test_eq(out["NUCLIDE"].tolist(), [28])
test_eq(out["UNIT"].tolist(), [12])
test_eq(out["LAB"].tolist(), [STATIC_METADATA["LAB"]])
print("MultiGateCB + RemapCB + SetValueCB: known values pass the gate, nomenclatures mapped, constant metadata assigned. ✓")

In [ ]:
# Verify MultiGateCB raises ValueError on an unmapped (typo'd) NUCLIDE value, before RemapCB ever runs
dfs_bad = {
    "SEAWATER": pd.DataFrame({
        "NUCLIDE": ["I129", "I-129_typo"],
        "UNIT": ["at_l", "at_l"],
    })
}
try:
    Transformer(dfs_bad, cbs=[
        MultiGateCB(lut_pairs={"NUCLIDE": NUCLIDE_LUT, "UNIT": UNIT_LUT}),
        RemapCB(lut=NUCLIDE_LUT, col_remap="NUCLIDE", col_src="NUCLIDE"),
        RemapCB(lut=UNIT_LUT, col_remap="UNIT", col_src="UNIT"),
    ])()
    raise AssertionError("Expected MultiGateCB to raise on an unmapped NUCLIDE value")
except ValueError as e:
    test_eq("I-129_typo" in str(e), True)
    print(f"MultiGateCB: correctly raised on unmapped NUCLIDE token before RemapCB ran — {e}")

In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB(),
    AddUnknownDepthCB(),
    MeltCB(META_COLS, VAL_COLS),
    ParseNuclideUnitCB(),
    AuditDropNullCB(["VALUE"]),
    MultiGateCB(lut_pairs={"NUCLIDE": NUCLIDE_LUT, "UNIT": UNIT_LUT}),
    RemapCB(lut=NUCLIDE_LUT, col_remap="NUCLIDE", col_src="NUCLIDE"),
    RemapCB(lut=UNIT_LUT, col_remap="UNIT", col_src="UNIT"),
    SetValueCB(constants=STATIC_METADATA),
])
tfm()

out = tfm.dfs["SEAWATER"]
print(out[["NUCLIDE", "UNIT", "LAB"]].drop_duplicates().to_string())

## Standardise final columns

`AuditDropNullCB(["TIME"])` logs and explicitly drops rows with a missing `TIME` before encoding, making the previously implicit `EncodeTimeCB` row-drop visible. Three shared callbacks then complete the pipeline:

- `SanitizeLonLatCB`: validates longitude/latitude ranges and ensures correct sign convention
- `EncodeTimeCB`: encodes the TIME column into the NetCDF-compatible numeric representation
- `AddSampleIDCB`: assigns a sequential `SMP_ID` and preserves the provider's `SMP_ID_PROVIDER`

All three are imported from `marisco.callbacks` and require no FS2025-specific configuration.

### Cast STATION to string before encoding

FS2025's `Station` column is pure numeric, so pandas infers `int64` — but `STATION` maps to
a `string`-typed NetCDF variable, and the NetCDF4 C library raises when the encoder tries to
write an `int64` value into a `string` variable. `FormatStationCB` casts `STATION` to `str` as
the last step before encoding.

This is the one column that still needs a manual cast. `SMP_ID_PROVIDER` is also a
`string`-typed variable, but `AddSampleIDCB(col_provider="SMP_ID_PROVIDER")` already casts it;
the enum columns (`NUCLIDE`, `UNIT`, `LAB`) and the `uint64` columns (`TIME`, `SMP_ID`) are
handled by `NetCDFEncoder` itself. The general `EnforceCDLSchemaCB` schema-casting framework
that used to cover all of this (helpers + a template-driven `CDL_CAST_LUT`) has been removed as
unused generality — `STATION` was the only column it was actually protecting that nothing else
in the pipeline already handles.

`FormatStationCB` is defined locally in this notebook only; it does not touch
`marisco/callbacks.py`.

In [ ]:
#| export
class FormatStationCB(PerGroupCB):
    "Cast STATION to str for the NetCDF4 string-typed station variable."
    grps = ["SEAWATER"]

    def each_grp(self, grp, df, tfm):
        df["STATION"] = df["STATION"].astype(str)

In [ ]:
# Verify FormatStationCB casts STATION to str
dfs_mock = {'SEAWATER': pd.DataFrame({'STATION': [341, 415]})}
tfm = Transformer(dfs_mock, cbs=[FormatStationCB()])
tfm()
out = tfm.dfs['SEAWATER']
test_eq(out['STATION'].tolist(), ['341', '415'])
test_eq(all(isinstance(v, str) for v in out['STATION']), True)  # what the encoder's per-element NetCDF write actually needs
print("FormatStationCB: STATION cast to str. ✓")

In [ ]:
#|eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB(),
    AddUnknownDepthCB(),
    MeltCB(META_COLS, VAL_COLS),
    ParseNuclideUnitCB(),
    AuditDropNullCB(["VALUE"]),
    MultiGateCB(lut_pairs={"NUCLIDE": NUCLIDE_LUT, "UNIT": UNIT_LUT}),
    RemapCB(lut=NUCLIDE_LUT, col_remap="NUCLIDE", col_src="NUCLIDE"),
    RemapCB(lut=UNIT_LUT, col_remap="UNIT", col_src="UNIT"),
    SetValueCB(constants=STATIC_METADATA),
    SanitizeLonLatCB(),
    AuditDropNullCB(["TIME"]),
    EncodeTimeCB(),
    AddSampleIDCB(col_provider="SMP_ID_PROVIDER"),
    FormatStationCB(),
])
tfm()
out = tfm.dfs['SEAWATER']
print(f"Final shape: {out.shape}")
print("Columns:", out.columns.tolist())
print(out[['SMP_ID', 'SMP_ID_PROVIDER', 'NUCLIDE', 'UNIT', 'LAB']].head(4).to_string())

In [ ]:
#|eval: false
print("Final data summary (uppercase columns only):")
upper_cols = [c for c in out.columns if c.isupper()]
print(out[upper_cols].describe().to_string())

Final data summary (uppercase columns only):
              LAT         LON          TIME  SMP_DEPTH        TEMP         SAL         VALUE           UNC  NUCLIDE   UNIT    LAB      SMP_ID
count  177.000000  177.000000  1.770000e+02      177.0  177.000000  177.000000  1.770000e+02  1.770000e+02    177.0  177.0  177.0  177.000000
mean    78.954240   -4.121274  1.754350e+09       -1.0    0.884859   33.794202  3.038391e+09  7.779234e+07     28.0   12.0    0.0   89.000000
std      0.412695    5.923451  3.917809e+05        0.0    2.362646    1.639711  1.330420e+09  3.403782e+07      0.0    0.0    0.0   51.239633
min     78.832167  -13.403667  1.753834e+09       -1.0   -1.815000   28.988500  1.502199e+08  4.745141e+06     28.0   12.0    0.0    1.000000
25%     78.833000   -8.998833  1.754006e+09       -1.0   -0.956300   33.023500  2.114304e+09  5.412238e+07     28.0   12.0    0.0   45.000000
50%     78.833333   -4.002000  1.754179e+09       -1.0    0.321000   34.699000  2.674898e+09  6.833433e

## NetCDF encoder

The encoder wraps the full pipeline and writes the standardised data to a NetCDF4 file. Global attributes are assembled via `GlobAttrsFeeder` with `BboxCB`, `DepthRangeCB`, `TimeRangeCB`, plus keywords and processing logs.

The resulting file contains spatial, depth, and time coverage derived from the transformed seawater data, together with FS2025 keywords and the recorded processing steps.

In [ ]:
#| exports
FS2025_KEYWORDS = [
    "Fram Strait","Greenland Sea","I-129","radionuclides","seawater","Arctic Ocean",
]

def get_attrs(tfm):
    "Retrieve global attributes for Fram Strait 2025."
    return GlobAttrsFeeder(tfm.dfs, cbs=[
        BboxCB(),
        DepthRangeCB(),
        TimeRangeCB(),
        KeyValuePairCB("keywords", ", ".join(FS2025_KEYWORDS)),
        KeyValuePairCB("publisher_postprocess_logs", ", ".join(tfm.logs)),
    ])()

In [ ]:
#| exports
def encode(fname_out=None  # Output NetCDF file path; defaults to fname_out
            ):
    "Encode Fram Strait 2025 data to NetCDF4."
    fname_out = fname_out or globals().get("fname_out", "Fram_Strait_2025.nc")
    dfs = load_data()
    tfm = Transformer(dfs, cbs=[
        RenameColsCB(),ParseDateTimeCB(),AddUnknownDepthCB(),
        MeltCB(META_COLS, VAL_COLS),
        ParseNuclideUnitCB(),
        AuditDropNullCB(["VALUE"]),
        MultiGateCB(lut_pairs={"NUCLIDE": NUCLIDE_LUT, "UNIT": UNIT_LUT}),
        RemapCB(lut=NUCLIDE_LUT, col_remap="NUCLIDE", col_src="NUCLIDE"),
        RemapCB(lut=UNIT_LUT, col_remap="UNIT", col_src="UNIT"),
        SetValueCB(constants=STATIC_METADATA),
        SanitizeLonLatCB(),
        AuditDropNullCB(["TIME"]),
        EncodeTimeCB(),AddSampleIDCB(col_provider="SMP_ID_PROVIDER"),
        FormatStationCB(),
    ])
    tfm()
    encoder = NetCDFEncoder(tfm.dfs, dest_fname=fname_out,
                            global_attrs=get_attrs(tfm))
    encoder.encode()

In [ ]:
#|eval: false
# Encode to NetCDF
encode("../../_data/output/fram_strait2025.nc")
print("Fram Strait 2025 NetCDF written.")

Fram Strait 2025 NetCDF written.


In [ ]:
#|eval: false
to_csv("../../_data/output/fram_strait2025.nc")

[Path('../../_data/output/fram_strait2025_SEAWATER.csv')]